# Task 1: Multi-Agent Design Thinking

## Business task

**Review a customer-support dataset, identify the main sources of dissatisfaction, and prepare a stakeholder-ready improvement brief.**

The workflow has three sequential stages: understand the data, interpret the patterns, and communicate the recommended actions.

## Agent roles

### 1. Data Quality Analyst

- **Role:** Data Quality Analyst
- **Goal:** Validate the support dataset and produce a reliable summary of its structure and quality.
- **Backstory:** You are a meticulous analytics engineer who checks schemas, missing values, duplicates, inconsistent labels, and suspicious records before anyone draws conclusions. You document assumptions and flag limitations so downstream analysis is based on trustworthy evidence.
- **Responsibility:** Inspect the dataset, clean or annotate quality issues, and report the fields and records that can safely be used for analysis.

### 2. Customer Insights Researcher

- **Role:** Customer Insights Researcher
- **Goal:** Discover the most important support themes, dissatisfaction drivers, and trends using the validated dataset.
- **Backstory:** You are an experienced customer researcher who combines quantitative analysis with practical product intuition. You look for meaningful patterns in ticket categories, sentiment, resolution time, and repeat contacts, while separating strong evidence from speculation.
- **Responsibility:** Analyze the validated data, rank the key findings, and connect each finding to supporting evidence. You do not redesign the report or make unsupported operational promises.

### 3. Stakeholder Communications Strategist

- **Role:** Stakeholder Communications Strategist
- **Goal:** Turn the research findings into a concise brief with prioritized, actionable recommendations.
- **Backstory:** You are a clear and persuasive business writer who regularly translates technical analysis for product and operations leaders. You focus on decisions, expected impact, ownership, and next steps while preserving the evidence and caveats supplied by the researchers.
- **Responsibility:** Draft the stakeholder-ready summary, prioritize recommendations, and explain the business rationale without changing the underlying findings.

## Why use multiple specialized agents?

Specialized agents can outperform one generalist because each agent has a narrower objective, distinct quality checks, and a useful handoff to the next stage; this reduces the chance that data validation, interpretation, and communication will be rushed or conflated. A single generalist may be better for a small, familiar dataset or a quick first draft, where coordination overhead and repeated context passing would cost more than specialization; human review is still needed for high-impact decisions.


# Task 2: Build Agents and Assign Tools

## Tool assignment

- **Data Quality Analyst:** `FileReadTool` is enough to inspect the supplied dataset and supporting files. This agent does not receive a search or writing tool because its job is to validate evidence, not gather unrelated information or publish conclusions.
- **Customer Insights Researcher:** `FileReadTool` allows targeted inspection of the validated dataset and quality report. The researcher receives only file access because the available legacy `CSVSearchTool` requires an unrelated `OPENAI_API_KEY`; analysis remains grounded in the project files while using the Gemini LLM.
- **Stakeholder Communications Strategist:** `FileWriterTool` lets the communications agent save the final improvement brief. It receives the earlier findings through task context and does not receive raw-data tools, which helps prevent it from changing the analysis while drafting.

Each agent uses a separate Gemini `LLM` configuration. The API key is read from the `GEMINI_API_KEY` environment variable after the user enters it at runtime and is never stored in the notebook.


In [1]:
import os
from getpass import getpass

from crewai import Agent, LLM
from crewai_tools import FileReadTool, FileWriterTool

# Prompt for the key on every execution without displaying or storing it in the notebook.
gemini_api_key = getpass("Enter your Gemini API key: ").strip()
if not gemini_api_key:
    raise ValueError("A Gemini API key is required to run this cell.")
os.environ["GEMINI_API_KEY"] = gemini_api_key

# Use the checked-in sample dataset in this notebook's folder.
support_data_file = "customer_support_sample.csv"
data_quality_tools = [FileReadTool(file_path=support_data_file)]
research_tools = [FileReadTool(file_path=support_data_file)]
communications_tools = [FileWriterTool()] 

# Separate LLM objects make each role's behavior explicit and independently tunable.
data_quality_llm = LLM(
    model="gemini/gemini-3.6-flash",
    temperature=0.0,
)
research_llm = LLM(
    model="gemini/gemini-3.6-flash",
    temperature=0.2,
)
communications_llm = LLM(
    model="gemini/gemini-3.6-flash",
    temperature=0.5,
)

# One iteration and no automatic retries keep quota failures from multiplying API calls.
agent_limits = {
    "max_iter": 1,
    "max_retry_limit": 0,
}

data_quality_agent = Agent(
    role="Data Quality Analyst",
    goal="Validate customer_support_sample.csv and document quality issues before analysis.",
    backstory=(
        "You are a meticulous analytics engineer. Inspect the provided customer-support CSV, "
        "check missing values, duplicates, inconsistent labels, and suspicious records, then "
        "document assumptions."
    ),
    tools=data_quality_tools,
    llm=data_quality_llm,
    verbose=True,
    **agent_limits,
)

research_agent = Agent(
    role="Customer Insights Researcher",
    goal="Find and rank evidence about dissatisfaction drivers in customer_support_sample.csv.",
    backstory=(
        "You are an experienced customer researcher who combines quantitative analysis with "
        "practical product intuition and separates evidence from speculation."
    ),
    tools=research_tools,
    llm=research_llm,
    verbose=True,
    **agent_limits,
)

communications_agent = Agent(
    role="Stakeholder Communications Strategist",
    goal="Turn validated findings into a concise, prioritized improvement brief for leaders.",
    backstory=(
        "You are a clear and persuasive business writer. You translate analysis into "
        "decisions, owners, expected impact, caveats, and practical next steps."
    ),
    tools=communications_tools,
    llm=communications_llm,
    verbose=True,
    **agent_limits,
)

agents = [data_quality_agent, research_agent, communications_agent]
print(f"Created {len(agents)} CrewAI agents with one-iteration, no-retry limits.")

Created 3 CrewAI agents with one-iteration, no-retry limits.



# Task 3: Define Tasks and Process

## Sequential task dependencies

The crew runs three tasks in order with `Process.sequential`:

1. The **data quality task** validates the customer-support data and returns a structured quality report.
2. The **customer insights task** receives that report through `context=[quality_task]`, then produces an evidence-based findings table.
3. The **communications task** receives both earlier outputs through `context=[quality_task, insights_task]` and drafts the stakeholder brief.

The expected outputs deliberately use stable headings and tables so that each downstream agent can reliably interpret the previous handoff.

## Handoff-format adjustment

During an initial design pass, the quality analyst returned general prose, but the researcher needed consistent fields such as issue type, affected records, severity, and evidence. I fixed the handoff by changing the quality task's `expected_output` to require a Markdown table with those exact columns, and I added the same required structure to the research task's prompt before passing its output to the communications agent.


In [2]:
import contextlib
import io

from crewai import Crew, Process, Task

quality_task = Task(
    description=(
        "Inspect the customer-support dataset and any available supporting project files. "
        "Validate the schema, missing values, duplicates, inconsistent labels, and suspicious "
        "records. Do not infer business conclusions before documenting data quality. Return "
        "a concise report with the required Markdown table columns: issue type, affected "
        "records or fields, severity, evidence, and recommended handling."
    ),
    expected_output=(
        "A Markdown data-quality report containing: (1) a short dataset overview, "
        "(2) a table with exactly these columns: issue type, affected records or fields, "
        "severity, evidence, recommended handling, and (3) explicit assumptions and limitations."
    ),
    agent=data_quality_agent,
)

insights_task = Task(
    description=(
        "Use the data-quality report from the previous task as a constraint. Analyze the "
        "validated customer-support data for dissatisfaction drivers and trends in ticket "
        "category, sentiment, resolution time, and repeat contacts. Separate measured "
        "evidence from hypotheses and do not use fields flagged as unreliable."
    ),
    expected_output=(
        "A Markdown customer-insights report with: (1) a table containing exactly these "
        "columns: finding, supporting evidence, affected segment, confidence, and business "
        "implication, (2) the top three prioritized findings, and (3) a short list of caveats."
    ),
    agent=research_agent,
    context=[quality_task],
)

communications_task = Task(
    description=(
        "Use the data-quality report and customer-insights report provided in context. "
        "Draft a stakeholder-ready improvement brief for product and operations leaders. "
        "Preserve the evidence, confidence levels, and caveats; do not invent metrics or "
        "rewrite findings that were marked unreliable."
    ),
    expected_output=(
        "A concise Markdown stakeholder brief containing: executive summary, prioritized "
        "recommendations, owner, rationale, expected impact, measurement plan, risks, "
        "and next steps. Every recommendation must cite the finding it addresses."
    ),
    agent=communications_agent,
    context=[quality_task, insights_task],
)

tasks = [quality_task, insights_task, communications_task]
crew = Crew(
    agents=agents,
    tasks=tasks,
    process=Process.sequential,
    verbose=True,
)

# Jupyter already has an active event loop, so use CrewAI's async kickoff API.
# Keep the complete partial log available even when an API or tool error stops the run.
execution_log_buffer = io.StringIO()
crew_result = None
crew_error = None
with contextlib.redirect_stdout(execution_log_buffer), contextlib.redirect_stderr(execution_log_buffer):
    try:
        crew_result = await crew.kickoff_async()
    except Exception as error:
        crew_error = error
execution_log = execution_log_buffer.getvalue()

print("=== FULL CREW EXECUTION LOG ===")
print(execution_log)
if crew_error:
    print("=== CREW EXECUTION REVIEW ===")
    print(f"The sequential run stopped before completion: {type(crew_error).__name__}: {crew_error}")
    print("Check the Gemini quota and confirm that the input dataset is available to FileReadTool.")
else:
    print("=== FINAL CREW OUTPUT ===")
    print(crew_result)
    print("=== OUTPUT REVIEW ===")
    for index, task_output in enumerate(crew_result.tasks_output, start=1):
        print(f"Task {index}: {task_output.agent}")
        print(task_output.raw)
        print("-")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0b435758-0254-449f-97e9-795eeabf51c1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Inspect the customer-support dataset and any available supporting project files. Validate the schema,    │
│  missing values, duplicates, inconsistent labels, and suspicious records. Do not infer business conclusions     │
│  before documenting data quality. Return a concise report with the required Markdown table columns: issue       │
│  type, affected records or fields, severity, evidence, and recommended handling.                                │
│  ID: 9ec6e2f5-09c7-49aa-bd2c-24b314da8fd2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Quality Analyst                                                                                    │
│                                                                                                                 │
│  Task: Inspect the customer-support dataset and any available supporting project files. Validate the schema,    │
│  missing values, duplicates, inconsistent labels, and suspicious records. Do not infer business conclusions     │
│  before documenting data quality. Return a concise report with the required Markdown table columns: issue       │
│  type, affected records or fields, severity, evidence, and recommended handling.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_a_files_content                                                                                     │
│  Args: {'file_path': 'customer_support_sample.csv', 'start_line': 1, 'line_count': 100}                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: read_a_files_content                                                                                     │
│  Output: ticket_id,category,sentiment,resolution_hours,repeat_contact,channel,customer_segment                  │
│  T001,Billing,negative,48,yes,email,SMB                                                                         │
│  T002,Login,negative,12,no,chat,Consumer                                                                        │
│  T003,Refund,negative,72,yes,email,Consumer                                                                     │
│  T004,Shipping,neutral,36,no,web,SMB                                                                            │
│  T005,Login,positive,2,no,chat,Consumer                                                                         │
│  T006,Billing,negative,60,yes,email,Enterprise                                                                  │
│  T007,Shipping,negative,96,yes,email,Consumer                                                                   │
│  T008,Account,neutral,18,no,web,SMB                                                                             │
│  T009,Refund,negative,80,yes,chat,Consumer                                                                      │
│  T010,Billing,positive,8,no,chat,Enterprise                                                                     │
│  T011,Login,negative,24,no,email,SMB                                                                            │
│  T012,Shipping,neutral,30,no,web,Enterprise                                                                     │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: Google Gemini API error: 429 - You exceeded your current quota, please check your plan and billing      │
│  details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To    │
│  monitor your current usage, head to: https://ai.dev/rate-limit.                                                │
│  * Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit:     │
│  20, model: gemini-3.6-flash                                                                                    │
│  Please retry in 38.35535776s.                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Inspect the customer-support dataset and any available supporting project files. Validate the schema,    │
│  missing values, duplicates, inconsistent labels, and suspicious records. Do not infer business conclusions     │
│  before documenting data quality. Return a concise report with the required Markdown table columns: issue       │
│  type, affected records or fields, severity, evidence, and recommended handling.                                │
│  Agent: Data Quality Analyst                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

=== FULL CREW EXECUTION LOG ===
Tool read_a_files_content executed with result: ticket_id,category,sentiment,resolution_hours,repeat_contact,channel,customer_segment
T001,Billing,negative,48,yes,email,SMB
T002,Login,negative,12,no,chat,Consumer
T003,Refund,negative,72,yes,email,C...
Maximum iterations reached. Requesting final answer.
ERROR:root:Google Gemini API error: 429 - You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash
Please retry in 38.35535776s.
ERROR:crewai.flow.runtime:Error executing listener ensure_force_final_answer: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. 

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 0b435758-0254-449f-97e9-795eeabf51c1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Task 4: Try Hierarchical Delegation

## Hierarchical design

The hierarchical crew adds a **Crew Manager** responsible for delegating the same customer-support workflow to the three specialists, checking their handoffs, and requesting corrections when an output misses the required format. The manager has its own Gemini LLM configuration and no raw-data tool because it coordinates and reviews work rather than replacing the data analyst.

The comparison runner uses the same task definitions and records elapsed time, token usage when CrewAI exposes it, output-quality checks, and reliability. It performs exactly one sequential run and one hierarchical run when `RUN_COMPARISON = True`; agent retries are disabled to prevent quota waste.

## Sequential vs. hierarchical

| Process | Pros | Cons | When to use |
|---|---|---|---|
| Sequential | Predictable order, explicit handoffs, lower coordination overhead, and usually lower latency and cost | No central reviewer; a weak early output can flow into later tasks | Use for stable pipelines with clear dependencies and well-defined outputs |
| Hierarchical | A manager can delegate dynamically, review quality, resolve ambiguity, and ask specialists to improve weak work | More manager calls, higher latency and token cost, and more moving parts that can fail | Use for ambiguous work, changing priorities, or tasks that benefit from active review |

## Current execution status

The notebook structure for both runs is complete, but the runs are **not yet demonstrated successfully** because the Gemini API quota was exhausted. Once the quota resets, run the agent setup cell, the task setup cell, this hierarchical setup cell, then set `RUN_COMPARISON = True` and execute it once. The output will contain the sequential and hierarchical latency, token usage, reliability, quality checks, errors, and full logs.

In [3]:
from time import perf_counter

# The manager delegates and reviews; it does not replace specialist file access.
manager_llm = LLM(
    model="gemini/gemini-3.6-flash",
    temperature=0.1,
)
manager_agent = Agent(
    role="Crew Manager",
    goal=(
        "Delegate the customer-support analysis to the appropriate specialists, review each "
        "handoff against its expected format, and deliver a coherent final brief."
    ),
    backstory=(
        "You are a disciplined delivery manager. You coordinate specialists, identify missing "
        "evidence or malformed outputs, request targeted corrections, and preserve caveats."
    ),
    llm=manager_llm,
    verbose=True,
    max_iter=1,
    max_retry_limit=0,
)

hierarchical_crew = Crew(
    agents=agents,
    tasks=tasks,
    manager_agent=manager_agent,
    process=Process.hierarchical,
    verbose=True,
)


def output_quality(result):
    """Return simple, repeatable checks for the shared task outputs."""
    if result is None:
        return {"completed_tasks": 0, "required_sections": 0}

    output_text = "\n".join(
        getattr(task_output, "raw", "") for task_output in result.tasks_output
    ).lower()
    required_sections = sum(
        section in output_text
        for section in ("evidence", "recommendation", "caveat", "next steps")
    )
    return {
        "completed_tasks": len(result.tasks_output),
        "required_sections": required_sections,
    }


def token_summary(result):
    """Extract token totals when supported by the installed CrewAI version."""
    usage = getattr(result, "token_usage", None)
    if usage is None:
        return "unavailable"
    if isinstance(usage, dict):
        return usage
    return {
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
    }


async def run_and_measure(label, selected_crew):
    """Run one crew once and retain its full log even when the API fails."""
    log_buffer = io.StringIO()
    started = perf_counter()
    result = None
    error = None
    with contextlib.redirect_stdout(log_buffer), contextlib.redirect_stderr(log_buffer):
        try:
            result = await selected_crew.kickoff_async()
        except Exception as caught_error:
            error = caught_error

    return {
        "process": label,
        "latency_seconds": round(perf_counter() - started, 2),
        "reliable": error is None,
        "error": None if error is None else f"{type(error).__name__}: {error}",
        "token_usage": token_summary(result),
        "quality": output_quality(result),
        "execution_log": log_buffer.getvalue(),
        "result": result,
    }


# One full sequential run and one full hierarchical run are the minimum comparison.
# Leave disabled until the Gemini quota is available; each enabled run consumes quota.
RUN_COMPARISON = False
comparison_results = []
if RUN_COMPARISON:
    comparison_results = [
        await run_and_measure("sequential", crew),
        await run_and_measure("hierarchical", hierarchical_crew),
    ]
    for comparison in comparison_results:
        print(f"=== {comparison['process'].upper()} REVIEW ===")
        print(f"Latency: {comparison['latency_seconds']} seconds")
        print(f"Reliable: {comparison['reliable']}")
        print(f"Token usage: {comparison['token_usage']}")
        print(f"Quality checks: {comparison['quality']}")
        print(f"Error: {comparison['error']}")
        print("Full execution log:")
        print(comparison["execution_log"])
else:
    print("Comparison is ready but disabled to protect the Gemini quota.")
    print("Set RUN_COMPARISON = True once, after the quota resets, to measure both runs.")

Comparison is ready but disabled to protect the Gemini quota.
Set RUN_COMPARISON = True once, after the quota resets, to measure both runs.


# Task 5: Evaluation and Cost Awareness

## Cost measurement

The comparison runner records `prompt_tokens`, `completion_tokens`, and `total_tokens` when the installed CrewAI version exposes them on `CrewOutput.token_usage`. Approximate cost is calculated from a configurable price card, because Gemini pricing varies by model, account, and billing tier. The code reports `unavailable` rather than inventing a cost when token usage is missing.

## Success criteria

1. **Factual grounding:** Claims are supported by `customer_support_sample.csv`, and unreliable fields are explicitly acknowledged.
2. **Completeness:** The output contains quality findings, prioritized insights, recommendations, caveats, owners, and next steps.
3. **Professional tone:** The final brief is concise, stakeholder-ready, actionable, and avoids unsupported certainty.

## Manual three-run scorecard

Scores are on a 1–5 scale, where 5 means the criterion was fully met. The table is intentionally marked **not scored** until three successful outputs exist; the earlier quota failures produced no valid business output and must not be scored as successful runs.

| Run | Factual grounding | Completeness | Professional tone | Status |
|---|---:|---:|---:|---|
| Sequential run 1 | Not scored | Not scored | Not scored | Requires a successful API run |
| Hierarchical run 2 | Not scored | Not scored | Not scored | Requires a successful API run |
| Single-agent Day 3 run 3 | Not scored | Not scored | Not scored | Requires the Day 3 output and a successful API run |

For this task, a multi-agent crew may be worth the added complexity when independent validation and structured handoffs materially improve the result. However, it costs more API calls than a single well-designed agent and is more exposed to quota or rate-limit failures. The local dataset is now available, but the Gemini quota still prevents claiming a reliable final business output. Until three successful runs are completed, a single-agent baseline is the lower-risk option for this notebook.

In [4]:
from decimal import Decimal

# Gemini pricing varies by account and billing tier; replace these placeholders with
# the current price card before interpreting the cost estimate.
PRICE_PER_MILLION_INPUT_TOKENS = Decimal("0")
PRICE_PER_MILLION_OUTPUT_TOKENS = Decimal("0")


def approximate_cost(token_usage):
    """Estimate USD cost from a CrewOutput token-usage mapping."""
    if token_usage == "unavailable" or not isinstance(token_usage, dict):
        return "unavailable"

    input_tokens = token_usage.get("prompt_tokens")
    output_tokens = token_usage.get("completion_tokens")
    if input_tokens is None or output_tokens is None:
        return "unavailable"

    cost = (
        Decimal(str(input_tokens)) / Decimal("1000000") * PRICE_PER_MILLION_INPUT_TOKENS
        + Decimal(str(output_tokens)) / Decimal("1000000") * PRICE_PER_MILLION_OUTPUT_TOKENS
    )
    return f"${cost:.6f}"


# Reuse the Task 4 runner and add an approximate cost field to each measured run.
async def evaluate_process(label, selected_crew):
    result = await run_and_measure(label, selected_crew)
    result["approximate_cost_usd"] = approximate_cost(result["token_usage"])
    return result


RUN_EVALUATION = False
if RUN_EVALUATION:
    evaluation_results = [
        await evaluate_process("sequential", crew),
        await evaluate_process("hierarchical", hierarchical_crew),
    ]
    for evaluation in evaluation_results:
        print(f"=== {evaluation['process'].upper()} COST REVIEW ===")
        print(f"Token usage: {evaluation['token_usage']}")
        print(f"Approximate cost: {evaluation['approximate_cost_usd']}")
        print(f"Latency: {evaluation['latency_seconds']} seconds")
        print(f"Reliable: {evaluation['reliable']}")
        print(f"Quality checks: {evaluation['quality']}")
        print(f"Error: {evaluation['error']}")
else:
    print("Evaluation utilities created. Set RUN_EVALUATION = True after entering current Gemini prices.")

Evaluation utilities created. Set RUN_EVALUATION = True after entering current Gemini prices.
